# Analytical workflow explained from repository scripts

**Manuscript:** *Contamination and taxon sampling explain conflicting eukaryote placements*

This notebook follows the paper’s logic in three parts:

| Part | Focus | Main displays | Principal `data/` directories |
| --- | --- | --- | --- |
| **A — Contamination landscape** | Distribution of contamination across archaeal MAGs (especially Asgard and candidate closest-relative lineages) | **Fig. 1**, Extended Data contamination figures | `data/decontamination/assessment/` |
| **B — Phylogenomic analysis** | Bias-controlled genome sets and trees; test of eukaryotic placement | **Fig. 2–3**, AU, CAT-GTR, post-hoc audit | `data/genome_sets/` · `data/taxonomic_counts/` · `data/PMS/` · `data/alignments/` · `data/trees/` · `data/decontamination/phylogenomic_sets/` · `data/supplementary_tables/` |
| **C — ESP inventories** | Whether contamination inflates functional (ESP) counts | **Fig. 4** | `data/ESP/` |

Scripts under `scripts/` are **example templates** (placeholder paths). Published numerical results are defined by the files under `data/` and the Supplementary / Extended Data tables.

**Repository:** https://github.com/tjcadd2020/Asgard-Eukaryote-Phylogenomics-2026  


## Genome accession tables (shared reference)

These lists define **which genomes** enter the study. They are used both for the contamination survey and for phylogenomics, but the **analytical goals differ** (Part A vs Part B).

| Genome set | Role | Accession + GTDB table |
| --- | --- | --- |
| GS-Zhang2025 | Imbalanced benchmark (Zhang et al. 2025) | `data/genome_sets/` · **Supplementary Table 1** |
| GS-Liu2021 | Imbalanced benchmark (Liu et al. 2021) | `data/genome_sets/` · **Supplementary Table 2** |
| GS-Zhang2025-B | Balanced version of GS-Zhang2025 | `data/genome_sets/` · **Supplementary Table 3** |
| GS-Liu2021-B | Balanced version of GS-Liu2021 | `data/genome_sets/` · **Supplementary Table 4** |
| GS-Present-B | Independently assembled balanced set (this study) | `data/genome_sets/` · **Supplementary Table 5** |

Assemblies were obtained from NCBI RefSeq/GenBank (and any additional public sources listed in those tables) using these accessions. Full raw FASTA collections are not re-hosted in this repository.


---

# Part A — Contamination landscape across archaeal MAGs

**Results section role:** first major results block of the paper.  
**Question:** Is contamination random, or concentrated in lineages relevant to eukaryogenesis?

This part does **not** resolve the tree of life. It establishes that contamination is pervasive, uneven and enriched in Asgard—especially in lineages repeatedly proposed as close relatives of eukaryotes—and that contaminant-derived proteins can appear eukaryote-like.


## A1 — Identification of candidate exogenous sequences in archaeal MAGs

### Purpose
Identify candidate exogenous sequences (bacterial, viral, eukaryotic or unresolved) within archaeal MAGs. The same detection procedure later supports decontamination for phylogenomics, but **here the product is a survey** for Fig. 1.

### Inputs

| Input | Detail | Documented in |
| --- | --- | --- |
| Representative archaeal MAGs | Asgard, TACK, Euryarchaeota, DPANN (and isolates where compared) | Accessions overlapping the public collections summarized with the study; lineage panels use the MAG sets underlying **Fig. 1** |
| Raw MAG FASTA (`*.fna`) | Contig/scaffold-level assemblies fetched from NCBI (and listed public sources) | Re-fetch via accessions; full FASTA not re-hosted |
| CAT + geNomad DBs | Contig taxonomy and viral detection | External DBs (see script) |

### Method (tools)
- **CAT** (sensitive): taxonomic classification of assembled sequences; exogenous candidates = unresolved / non-cellular / non-Archaea at the criteria in Methods  
- **geNomad** (conservative): viral contigs


### Outputs of A1 (detection products)

| Output | Content | Role |
| --- | --- | --- |
| Classification tables | CAT taxonomy (+ names/summaries) per MAG | Input to frequency tallies |
| Viral calls | geNomad conservative viral contigs | Input to frequency tallies |
| Flagged exogenous sequence lists | Sequences meeting Methods criteria for non-archaeal / unresolved origin | Input to **A2–A3** summaries and later to Part B cleaning |

Genome-level frequencies, Asgard lineage patterns and contaminant eukaryote-like proteins are **results of A2–A3**, not of detection alone.

### Example template script
`scripts/decontamination/run_decontamination.sh` (CAT + geNomad; Methods: identification of candidate exogenous sequences)

### Full script


```bash
#!/bin/bash
# ============================================================
# Script: run_decontamination.sh (EXAMPLE TEMPLATE)
# Purpose: Identify candidate exogenous sequences in archaeal MAGs 
#          and generate decontaminated (clean) genome collections.
#          This script follows the procedure described in the Methods section:
#          "Identification of candidate exogenous sequences".
# Tools: CAT v6.0 + geNomad v1.8.0
# Stage: decontamination (Main decontamination procedure)
# Author: Weili Lin
# Date: 2026-07-07
# Note: This is an EXAMPLE template only. All commands are placeholders.
# ============================================================

set -euo pipefail

# ------------------ Configuration (PLEASE MODIFY) ------------------
WORK_DIR="/path/to/your/project"
SORTWARE_DIR="/path/to/your/software"
RAW_GENOMES_DIR="${WORK_DIR}/data/genomes/raw_MAGs"          # Input: original MAGs
OUTPUT_DIR="${WORK_DIR}/results/decontamination"             # Output: decontaminated genomes

mkdir -p "${OUTPUT_DIR}/cat_classification" \
         "${OUTPUT_DIR}/genomad" \
         "${OUTPUT_DIR}/logs"

echo "=== [EXAMPLE] Starting main decontamination workflow ==="
echo "Input raw genomes : ${RAW_GENOMES_DIR}"
date
echo ""

# ============================================================
# Contig-level taxonomic classification using CAT v6.0
# ============================================================
echo ">>> Running CAT classification (sensitive mode)..."
CAT_pack contigs \
	-c "${RAW_GENOMES_DIR}/example_genome.fna" \
	-d "${SORTWARE_DIR}/20240422_CAT_nr/db" \
	-t "${SORTWARE_DIR}/20240422_CAT_nr/tax" \ 
    -o "${OUTPUT_DIR}/cat_classification/CAT_pack/contigs/example_genome" \
    --sensitive -n 18


CAT_pack add_names \
	-i "${OUTPUT_DIR}/cat_classification/CAT_pack/contigs/example_genome.contig2classification.txt" \
	-o "${OUTPUT_DIR}/cat_classification/CAT_pack/add_names/example_genome.official_names.txt" \ 
    -t "${SORTWARE_DIR}/20240422_CAT_nr/tax" \
	--only_official


CAT_pack summarise \
	-c "${RAW_GENOMES_DIR}/example_genome.fna" \
    -i "${OUTPUT_DIR}/cat_classification/CAT_pack/contigs/example_genome.contig2classification.txt" \
    -o "${OUTPUT_DIR}/cat_classification/CAT_pack/add_names/example_genome.summary.txt"


echo "    (CAT classification step - placeholder)"
echo ""

# ============================================================
# Independent viral detection using geNomad v1.8.0
# ============================================================
echo ">>> Running geNomad for viral sequence detection (conservative mode)..."
genomad end-to-end \
	--threads 4 \
	--conservative \
	"${RAW_GENOMES_DIR}/example_genome.fna" \
    "${OUTPUT_DIR}/genomad" \
    "${SORTWARE_DIR}/genomad_db"


echo "    (geNomad viral detection step - placeholder)"
echo ""
```

## A2 — Contamination is uneven across archaea and enriched in Asgard

### Purpose
Turn contig-level flags from **A1** into **genome-level frequencies** and compare major groups. This is the first results claim of the paper: contamination is not background noise.

### Inputs
- Exogenous-sequence calls from A1 (CAT + geNomad) for MAGs assigned to Asgard, TACK, Euryarchaeota and DPANN
- Optional isolate-vs-MAG contrast for a ≥1% burden summary statistic (not used as an automatic removal rule)

### Outputs

| Output | Content | Where |
| --- | --- | --- |
| Frequencies by major group | % MAGs with bacterial / viral / eukaryotic / unclassified contaminant contigs | `data/decontamination/assessment/` → **Fig. 1a** |
| Isolate vs MAG burden (e.g. ≥1% of sequence) | Summary only | Extended Data panels under assessment (e.g. Supplementary/Extended Fig. on 1% threshold) |

### Result to report
Asgard MAGs show the **highest** contamination frequencies across contaminant categories relative to TACK, Euryarchaeota and DPANN (Fig. 1a). MAGs are far more often above a 1% burden than pure-culture complete genomes.


## A3 — Candidate closest-relative lineages and contaminant eukaryote-like proteins

### Purpose
Ask whether contamination is further concentrated **inside** Asgard—especially in lineages repeatedly proposed as close relatives of eukaryotes—and whether contaminant sequence can carry **eukaryote-like** proteins that would bias affinity.

### Inputs
- Same A1 exogenous-sequence calls, restricted to Asgard MAGs with class- and order-level GTDB labels
- Proteins encoded on sequences flagged as exogenous, scored for apparent similarity to eukaryotic homologues

### Outputs

| Output | Content | Where |
| --- | --- | --- |
| Contamination by Asgard **class / order** | Frequencies per lineage and contaminant type | `data/decontamination/assessment/` → **Fig. 1b**, Extended Data Figs. 2–3 |
| Contaminant-derived **eukaryote-like** proteins | Detection rates by Asgard lineage | `data/decontamination/assessment/` → **Fig. 1c**, Extended Data Figs. 4–5 |

### Result to report
Lineages such as **Hodarchaeales** and **Njordarchaeales** (and related groups) rank among the most contaminated Asgard lineages. Contaminant-derived proteins with apparent eukaryotic similarity are enriched in the same groups—providing a mechanism for **misleading** archaeal–eukaryotic phylogenetic signal and for inflated ESP-like counts (Part C).

---

### End of Part A

Part A establishes the **contamination landscape**. It does **not** place eukaryotes on the tree.  
**Part B** starts phylogenomics: decontaminate each **GS**, balance sampling, build PMSs, and infer trees (Fig. 2–3).


---

# Part B — Phylogenomic analysis (tree of life)

**Results section role:** from “Sampling imbalance and contamination have distinct effects” through topology robustness.  
**Question:** Once contamination and sampling imbalance are controlled, where do eukaryotes sit relative to Asgard and TACK?

Part B **reuses** decontamination tools, but the unit of analysis is the **defined genome sets (GS)** and the **factorial design** (raw/clean × imbalanced/balanced × four PMSs).


### Data directories used in Part B

| Directory | Contents |
| --- | --- |
| `data/genome_sets/` | Accession lists + GTDB taxonomy for each GS (raw membership; balanced GS-*-B) |
| `data/taxonomic_counts/` | Phylum / class / order counts after hierarchical balancing |
| `data/PMS/` | Marker membership and overlap among the four PMSs |
| `data/alignments/` | Concatenated amino-acid supermatrices for GS × PMS × (raw\|clean) |
| `data/trees/maximum_likelihood/` | IQ-TREE consensus trees (`.contree`) |
| `data/trees/PMSF/` | LG+C60+F+G+PMSF sensitivity trees |
| `data/trees/CAT-GTR/` | PhyloBayes CAT-GTR chain outputs |
| `data/trees/AU_tests/` | Constraint topologies and AU test results |
| `data/trees/robustness_analyses.xlsx` | Cross-analysis topology summary (if deposited) |
| `data/decontamination/phylogenomic_sets/` | Contigs-removed (%) for primary decontamination of each GS; post-hoc audit summaries |
| `data/supplementary_tables/` | Publication tables (e.g. Tables 1–7, 18–20) linked to Part B |



## B1 — Decontamination of phylogenomic genome collections

### Purpose
Produce **clean** versions of each GS used for concatenation and tree search, and record how much sequence was removed **in those collections**. This is operational preparation for Fig. 2, not the Fig. 1 survey.

### Inputs

| Input | Detail | Documented in |
| --- | --- | --- |
| GS membership | Which accessions belong to GS-Zhang2025, GS-Liu2021, GS-Present-B and balanced variants | **Supplementary Tables 1–5**, `data/genome_sets/` |
| Raw FASTA for those accessions | Same download sources as above, restricted to each GS list | Local raw genome directories used for phylogenomics |
| CAT / geNomad calls | Contig flags from the detection workflow | Applied **per GS** to write clean FASTA |

### Outputs (Part B products)

| Output | Content | Where |
| --- | --- | --- |
| GS-*-**raw** vs GS-*-**clean** genome sets | Parallel collections for the factorial design | Used in all ML combinations; membership still Tables 1–5 |
| **Contigs removed (%)** under primary decontamination | Per phylogenomic collection (not the Fig. 1 frequency bars) | `data/decontamination/phylogenomic_sets/` → **Extended Data Table 4** (e.g. GS-Zhang2025-B-clean **4.91%**, GS-Liu2021-B-clean **4.22%**, GS-Present-B-clean **5.11%**) |

Moderate removal (~4–5%) is the level argued in the text to clear artefactual signal without stripping most of each genome.

### Script
Same template family as A1 (`run_decontamination.sh`), applied to GS FASTA lists rather than to the survey narrative of Fig. 1.


## B2 — Hierarchical taxonomic balancing

### Purpose
Build **GS-*-B** collections so Asgard ≈ TACK representation (Euryarchaeota/DPANN slightly lower), isolating sampling imbalance from contamination.

### Inputs

| Input | Where |
| --- | --- |
| GTDB-Tk summaries + quality-filtered candidates | Collection-building stage |
| **`sampling.xlsx`** | Retention choices (class→species, `n_retain`) with `scripts/balancing/` |
| Isolate metadata (GTDB R220) | External release files referenced in the R script |

### Outputs

| Output | Where |
| --- | --- |
| Balanced accession lists | **Supplementary Tables 3–5**, `data/genome_sets/GS-*-B*` |
| Phylum/class/order counts | `data/taxonomic_counts/` → **Supplementary Tables 18–20** |

### Script path
`scripts/balancing/hierarchical_balancing_TACK_GS-ThisStudy-Balanced.R`

### Full script


```r
################################################################################
# Hierarchical taxonomic balancing for TACK archaea (GS-Present-B)
# 
# Core logic (exactly as implemented and described in Methods):
# 1. At a given taxonomic rank, count the number of genomes in each lineage
# 2. Sort lineages from smallest to largest
# 3. average = remaining_target / number_of_lineages_still_to_be_processed
# 4. If a lineage has ≤ average genomes → keep ALL of them
# 5. If a lineage has > average genomes → go down to the next lower rank
#    (order → family → genus → species) and repeat the same calculation
# 6. Final decided numbers are recorded in sampling.xlsx
# 7. The last part of the script executes the sampling according to sampling.xlsx
################################################################################

##############################
# Stage 1: Data preparation
##############################

# Load GTDB release 220 metadata
# Used to extract high-quality isolate genomes at the complete-genome level
ar53_metadata_r220 <- read.delim(
  "/public/home/bdpguest/zhuruixin/software/release220/ar53_metadata_r220.tsv",
  stringsAsFactors = FALSE, header = TRUE
)

# Keep only high-quality RefSeq isolate complete genomes
ar53_metadata_r220_RS <- ar53_metadata_r220[
  substr(ar53_metadata_r220$accession, 1, 3) == "RS_" &
  ar53_metadata_r220$ncbi_genome_category == "none" &
  ar53_metadata_r220$ncbi_assembly_level == "Complete Genome" &
  ar53_metadata_r220$checkm2_completeness >= 70 &
  ar53_metadata_r220$checkm2_contamination <= 10, 
]

# Extract Thermoproteota isolates (part of TACK)
phylum <- sapply(ar53_metadata_r220_RS$gtdb_taxonomy, function(x) strsplit(x, ";")[[1]][2])
RS_Thermoproteota <- ar53_metadata_r220_RS[phylum == "p__Thermoproteota", ]
RS_Thermoproteota_class <- sapply(RS_Thermoproteota$gtdb_taxonomy, function(x) strsplit(x, ";")[[1]][3])

# Load GTDB-Tk classification results for MAGs at all assembly levels
# (contig-level, scaffold-level, chromosome-level and complete-genome level)
gtdbtk.ar53_contig          <- read.table("/public/home/bdpguest/zhuruixin/gtdbtk/MAG_archaea_contig_new/gtdbtk.ar53.summary.tsv",
                                          stringsAsFactors = FALSE, sep = "\t", header = TRUE)
gtdbtk.ar53_scaffold        <- read.table("/public/home/bdpguest/zhuruixin/gtdbtk/MAG_archaea_scaffold_new/gtdbtk.ar53.summary.tsv",
                                          stringsAsFactors = FALSE, sep = "\t", header = TRUE)
gtdbtk.ar53_chromosome      <- read.table("/public/home/bdpguest/zhuruixin/gtdbtk/MAG_archaea_chromosome_new/gtdbtk.ar53.summary.tsv",
                                          stringsAsFactors = FALSE, sep = "\t", header = TRUE)
gtdbtk.ar53_complete_genomes <- read.table("/public/home/bdpguest/zhuruixin/gtdbtk/MAG_archaea_complete_genomes/gtdbtk.ar53.summary.tsv",
                                          stringsAsFactors = FALSE, sep = "\t", header = TRUE)

gtdbtk.ar53.summary_all <- rbind(
  gtdbtk.ar53_contig,
  gtdbtk.ar53_scaffold,
  gtdbtk.ar53_chromosome,
  gtdbtk.ar53_complete_genomes
)

# Keep only genomes that survived dRep dereplication
dRep <- dir("/public/home/bdpguest/zhuruixin/dRep/Thermoproteota/dereplicated_genomes")
dRep_ids <- sapply(dRep, function(x) strsplit(x, ".fna")[[1]][1])
gtdbtk.ar53.summary <- subset(gtdbtk.ar53.summary_all, user_genome %in% dRep_ids)

# Parse class-level taxonomy (this is the starting rank for balancing)
gtdbtk.ar53.summary_class <- sapply(gtdbtk.ar53.summary$classification, function(x) strsplit(x, ";")[[1]][3])

##############################
# Stage 2: Exploration & decision phase
# (This is the part you did manually to decide the numbers in sampling.xlsx)
# The long chain of subsetting below is the concrete implementation of:
#   “look at counts → calculate average → keep small lineages fully →
#    drill down into large lineages at the next lower rank”
##############################

# ----- Class: Methanomethylicia -----
Methanomethylicia <- gtdbtk.ar53.summary[gtdbtk.ar53.summary_class == "c__Methanomethylicia", ]
Methanomethylicia_order <- sapply(Methanomethylicia$classification, function(x) strsplit(x, ";")[[1]][4])

# Order B29-G17 → Family DSZF01 → Species level
B29_G17 <- Methanomethylicia[Methanomethylicia_order == "o__B29-G17", ]
B29_G17_family <- sapply(B29_G17$classification, function(x) strsplit(x, ";")[[1]][5])
DSZF01 <- B29_G17[B29_G17_family == "f__DSZF01", ]
DSZF01_species <- sapply(DSZF01$classification, function(x) strsplit(x, ";")[[1]][7])

# Order Nezhaarchaeales
Nezhaarchaeales <- Methanomethylicia[Methanomethylicia_order == "o__Nezhaarchaeales", ]
Nezhaarchaeales_family <- sapply(Nezhaarchaeales$classification, function(x) strsplit(x, ";")[[1]][5])

B40_G2 <- Nezhaarchaeales[Nezhaarchaeales_family == "f__B40-G2", ]
B40_G2_genus <- sapply(B40_G2$classification, function(x) strsplit(x, ";")[[1]][6])
unknown <- B40_G2[B40_G2_genus == "g__", ]
unknown_species <- sapply(unknown$classification, function(x) strsplit(x, ";")[[1]][7])

WYZ_LMO8 <- Nezhaarchaeales[Nezhaarchaeales_family == "f__WYZ-LMO8", ]
WYZ_LMO8_genus <- sapply(WYZ_LMO8$classification, function(x) strsplit(x, ";")[[1]][6])
WYZ_LMO8_2 <- WYZ_LMO8[WYZ_LMO8_genus == "g__WYZ-LMO8", ]
WYZ_LMO8_2_species <- sapply(WYZ_LMO8_2$classification, function(x) strsplit(x, ";")[[1]][7])

# Order Methanomethylicales → Family Methanomethylicaceae → Genus → Species
Methanomethylicales <- Methanomethylicia[Methanomethylicia_order == "o__Methanomethylicales", ]
Methanomethylicales_family <- sapply(Methanomethylicales$classification, function(x) strsplit(x, ";")[[1]][5])
Methanomethylicaceae <- Methanomethylicales[Methanomethylicales_family == "f__Methanomethylicaceae", ]
Methanomethylicaceae_genus <- sapply(Methanomethylicaceae$classification, function(x) strsplit(x, ";")[[1]][6])

WYZ_LMO11 <- Methanomethylicaceae[Methanomethylicaceae_genus == "g__WYZ-LMO11", ]
WYZ_LMO11_species <- sapply(WYZ_LMO11$classification, function(x) strsplit(x, ";")[[1]][7])

Methanomethylicus <- Methanomethylicaceae[Methanomethylicaceae_genus == "g__Methanomethylicus", ]
Methanomethylicus_species <- sapply(Methanomethylicus$classification, function(x) strsplit(x, ";")[[1]][7])

WYZ_LMO10 <- Methanomethylicaceae[Methanomethylicaceae_genus == "g__WYZ-LMO10", ]
WYZ_LMO10_species <- sapply(WYZ_LMO10$classification, function(x) strsplit(x, ";")[[1]][7])

Methanosuratincola <- Methanomethylicaceae[Methanomethylicaceae_genus == "g__Methanosuratincola", ]
Methanosuratincola_species <- sapply(Methanosuratincola$classification, function(x) strsplit(x, ";")[[1]][7])

# ----- Class: Nitrososphaeria_A -----
Nitrososphaeria_A <- gtdbtk.ar53.summary[gtdbtk.ar53.summary_class == "c__Nitrososphaeria_A", ]
Nitrososphaeria_A_family <- sapply(Nitrososphaeria_A$classification, function(x) strsplit(x, ";")[[1]][5])

# ... (Caldarchaeaceae, HR02, Wolframiiraptoraceae and their genera/species)
# The same pattern continues: look at counts → decide whether to keep all
# or go one level deeper.

# ----- Class: Thermoprotei -----
Thermoprotei <- gtdbtk.ar53.summary[gtdbtk.ar53.summary_class == "c__Thermoprotei", ]
Thermoprotei_order <- sapply(Thermoprotei$classification, function(x) strsplit(x, ";")[[1]][4])
# ... continue the same hierarchical inspection

# ----- Class: Bathyarchaeia -----
Bathyarchaeia <- gtdbtk.ar53.summary[gtdbtk.ar53.summary_class == "c__Bathyarchaeia", ]
Bathyarchaeia_order <- sapply(Bathyarchaeia$classification, function(x) strsplit(x, ";")[[1]][4])
# ... many orders (EX4484-135, B25, B24, RBG-16-48-13, TCS64, etc.)
# Each large order is further inspected at family → genus → species

# ----- Class: Nitrososphaeria -----
Nitrososphaeria <- gtdbtk.ar53.summary[gtdbtk.ar53.summary_class == "c__Nitrososphaeria", ]
Nitrososphaeria_order <- sapply(Nitrososphaeria$classification, function(x) strsplit(x, ";")[[1]][4])
# ... Conexivisphaerales and Nitrososphaerales further broken down

# After finishing all the above inspections, the decided numbers
# (how many genomes to keep from each terminal lineage) are written
# into sampling.xlsx (sheet 5 in this case).

##############################
# Stage 3: Execution phase
# Read the pre-decided sampling table and actually sample the genomes
##############################

# Pre-compute taxonomy vectors for fast matching
class_gtdbtk   <- sapply(gtdbtk.ar53.summary$classification, function(x) strsplit(x, ";")[[1]][3])
order_gtdbtk   <- sapply(gtdbtk.ar53.summary$classification, function(x) strsplit(x, ";")[[1]][4])
family_gtdbtk  <- sapply(gtdbtk.ar53.summary$classification, function(x) strsplit(x, ";")[[1]][5])
genus_gtdbtk   <- sapply(gtdbtk.ar53.summary$classification, function(x) strsplit(x, ";")[[1]][6])
species_gtdbtk <- sapply(gtdbtk.ar53.summary$classification, function(x) strsplit(x, ";")[[1]][7])

library(xlsx)
sampling <- read.xlsx("sampling.xlsx", 5, header = FALSE)
# Columns: class | order | family | genus | species | n_retain

# Perform the actual sampling according to the decisions in sampling.xlsx
selected <- unlist(sapply(1:nrow(sampling), function(x) {
  print(x)   # progress indicator
  
  n_levels <- sum(!is.na(sampling[x, 1:5]))
  
  if (n_levels == 1) {
    pool <- gtdbtk.ar53.summary$user_genome[class_gtdbtk == sampling[x, 1]]
  } else if (n_levels == 2) {
    pool <- gtdbtk.ar53.summary$user_genome[
      class_gtdbtk == sampling[x, 1] & order_gtdbtk == sampling[x, 2]
    ]
  } else if (n_levels == 3) {
    pool <- gtdbtk.ar53.summary$user_genome[
      class_gtdbtk == sampling[x, 1] & order_gtdbtk == sampling[x, 2] &
      family_gtdbtk == sampling[x, 3]
    ]
  } else if (n_levels == 4) {
    pool <- gtdbtk.ar53.summary$user_genome[
      class_gtdbtk == sampling[x, 1] & order_gtdbtk == sampling[x, 2] &
      family_gtdbtk == sampling[x, 3] & genus_gtdbtk == sampling[x, 4]
    ]
  } else if (n_levels == 5) {
    pool <- gtdbtk.ar53.summary$user_genome[
      class_gtdbtk == sampling[x, 1] & order_gtdbtk == sampling[x, 2] &
      family_gtdbtk == sampling[x, 3] & genus_gtdbtk == sampling[x, 4] &
      species_gtdbtk == sampling[x, 5]
    ]
  }
  
  # Sample the pre-decided number (or take all if fewer are available)
  n_keep <- sampling[x, 6]
  if (length(pool) <= n_keep) {
    return(pool)
  } else {
    return(sample(pool, n_keep))
  }
}))

# Copy the selected genome files to the final directory
file.copy(
  paste0("/public/home/bdpguest/zhuruixin/dRep/Thermoproteota/dereplicated_genomes/",
         selected, ".fna"),
  "/public/home/bdpguest/zhuruixin/genome_all/Thermoproteota_MAGs_selected_167/"
)
```

## B3 — Four phylogenetic marker sets (PMS)

### Purpose
Independently curated markers **after** decontamination; **consistency across four PMSs** = primary robustness criterion for a placement.

| PMS | Source | n |
| --- | --- | --- |
| PMS-Isolate | Complete isolates only | 35 |
| PMS-HighMAG1 | + high-quality MAGs | 34 |
| PMS-HighMAG2 | + complete-level MAGs | 32 |
| PMS-MediumMAG | + CheckM medium-quality MAGs | 30 |

Shared core **28** markers → **Extended Data Table 1**, `data/PMS/PMS_composition_and_overlap.xlsx`.  
Single-protein trees used to drop HGT-like families before concatenation (Extended Data Fig. 6).


## B4 — Maximum-likelihood phylogenomics (IQ-TREE 3)

### Purpose
Factorial ML: every **GS × PMS × (raw|clean)** under `LG+C60+F+G` → **Fig. 2**.

### Inputs

| Input | Where |
| --- | --- |
| Genomes | Tables 1–5 / `data/genome_sets/` (raw or clean FASTA) |
| Markers | Extended Data Table 1 / `data/PMS/` |

### Outputs

| Output | Where |
| --- | --- |
| Supermatrices | `data/alignments/{GS}-{raw\|clean}_{PMS}.faa` |
| Consensus trees | `data/trees/maximum_likelihood/*.contree` |
| Topology across all combinations | **Supplementary Table 6** |

### Script path
`scripts/phylogenomics/run_iqtree_ml.sh`

### Full script


```bash
#!/bin/bash
# ============================================================
# Script: run_iqtree_ml.sh (EXAMPLE TEMPLATE)
# Purpose: Maximum-likelihood phylogenomic inference using IQ-TREE 3
#          under the LG+C60+F+G model with ultrafast bootstrap.
#          This follows the procedure described in Methods:
#          "Phylogenomic analyses".
# Stage: phylogenomics
# Author: Weili Lin
# Date: 2026-07-07
# Software: IQ-TREE 3
# Note: This is an EXAMPLE template only.
# ============================================================

set -euo pipefail

# ------------------ Configuration (PLEASE MODIFY) ------------------
WORK_DIR="/path/to/your/project"
SORTWARE_DIR="/path/to/your/software"
ALIGNMENT_DIR="${WORK_DIR}/data/alignments"
OUTPUT_DIR="${WORK_DIR}/results/trees"
MODEL="LG+C60+F+G"

mkdir -p "${OUTPUT_DIR}"

echo "=== [EXAMPLE] Starting IQ-TREE 3 Maximum-Likelihood analysis ==="
echo "Alignment directory : ${ALIGNMENT_DIR}"
echo "Output directory    : ${OUTPUT_DIR}"
echo "Model               : ${MODEL}"
date
echo ""

# ============================================================
# Step 1: Multiple sequence alignment (MAFFT-linsi)
# ============================================================
echo ">>> [Step 1] Running MAFFT-linsi for alignment..."
# For each single-gene marker set:

mafft-linsi --thread 6 \
	"${ALIGNMENT_DIR}/example_marker.faa" > 
    "${ALIGNMENT_DIR}/example_marker.aln"
	
echo "    (MAFFT alignment step - placeholder)"
echo ""

# ============================================================
# Step 2: Trimming ambiguously aligned regions (BMGE)
# ============================================================
echo ">>> [Step 2] Trimming alignments with BMGE (BLOSUM30 matrix)..."
# BMGE removes poorly aligned positions.

java -Xmx5000M -jar "${SORTWARE_DIR}/BMGE-1.12/BMGE.jar" \
	-i "${ALIGNMENT_DIR}/example_marker.aln" \
    -t AA -m BLOSUM30 \
	-of "${ALIGNMENT_DIR}/example_marker.trimmed.aln"


echo "    (BMGE trimming step - placeholder)"
echo ""

# ============================================================
# Step 3: Concatenation of trimmed alignments
# ============================================================
echo ">>> [Step 3] Concatenating trimmed single-gene alignments..."
# All trimmed markers are concatenated into a supermatrix.

${SORTWARE_DIR}/catfasta2phyml.pl \
	-f "${ALIGNMENT_DIR}/*.trimmed.aln" \
	--concatenate > "${ALIGNMENT_DIR}/supermatrix.faa"

echo "    (Concatenation step - placeholder)"
echo ""

# ============================================================
# Step 4: Maximum-likelihood inference with IQ-TREE 3
# ============================================================
echo ">>> [Step 4] Running IQ-TREE 3 (LG+C60+F+G + UFBoot)..."
# This is the core phylogenetic inference step.
# Model: LG+C60+F+G (site-heterogeneous mixture model)
# Bootstrap: 1000 ultrafast bootstrap replicates

mpirun --bind-to core --map-by ppr:2:node:PE=48 \
	${SORTWARE_DIR}/iqtree3-mpi \
	-s "${ALIGNMENT_DIR}/supermatrix.faa" \
	-st AA \
    -m ${MODEL} \
    -pre "${OUTPUT_DIR}/supermatrix" \
    -nt 48 \
    -bb 1000

echo "    (IQ-TREE 3 ML inference step - placeholder)"
echo ""

# ============================================================
# Completion
# ============================================================
echo "=== [EXAMPLE] IQ-TREE 3 Maximum-Likelihood analysis template completed ==="
echo ""
echo "Key parameters used in the study:"
echo "  - Model: LG+C60+F+G"
echo "  - Bootstrap: 1000 ultrafast bootstrap (UFBoot)"
echo "  - Software: IQ-TREE 3"
echo ""
echo "Note: This is an EXAMPLE template. All commands are placeholders."
echo "See Methods section: 'Phylogenomic analyses' for full details."
date
```

## B5 — AU topology tests

### Inputs
Full-control alignment from `data/alignments/` (e.g. GS-*-B-clean × PMS) + constraint tree list.

### Outputs
`data/trees/AU_tests/` → **Fig. 3d**, row in **Extended Data Table 5**.

### Script
`scripts/phylogenomics/run_au_topology_tests.sh`


```bash
#!/bin/bash
# ============================================================
# Script: run_au_topology_tests.sh (EXAMPLE TEMPLATE)
# Purpose: Approximately Unbiased (AU) topology tests to evaluate
#          competing hypotheses of eukaryotic placement.
#          This follows the procedure described in Methods:
#          "Topology testing and Bayesian inference".
# Stage: phylogenomics
# Author: Weili Lin
# Date: 2026-07-07
# Software: IQ-TREE 3
# Note: This is an EXAMPLE template only.
# ============================================================

set -euo pipefail

# ------------------ Configuration (PLEASE MODIFY) ------------------
WORK_DIR="/path/to/your/project"
ALIGNMENT="${WORK_DIR}/data/alignments/supermatrix.faa"
TREE_DIR="${WORK_DIR}/results/trees"
OUTPUT_DIR="${WORK_DIR}/results/au_tests"

mkdir -p "${OUTPUT_DIR}"

echo "=== [EXAMPLE] Starting AU Topology Tests ==="
echo "Alignment file : ${ALIGNMENT}"
echo "Output directory: ${OUTPUT_DIR}"
date
echo ""

# ============================================================
#  Run AU tests in IQ-TREE 3
# ============================================================
echo ">>> Running Approximately Unbiased (AU) tests..."

# IQ-TREE command structure for AU test:
# Example command (commented out):
iqtree3 \
  -s "${ALIGNMENT}" -st AA \
  -z "${TREE_DIR}/all_constraint_topologies.treels" \
  -zb 10000 -n 0 \
  -zw -au -m LG+C60+F+G4 -nt 64 \
  -pre "${OUTPUT_DIR}/AU_test_results"


echo "    (AU test execution - placeholder)"
echo ""
```

## B6 — Bayesian CAT-GTR (10 chains)

### Inputs
Full-control alignment (same class as B5).

### Outputs
`data/trees/CAT-GTR/`; per-chain summary in **Extended Data Table 5** (e.g. 7/10 chains: eukaryotes sister to TACK–Asgard); **Fig. 3e**.

### Script
`scripts/phylogenomics/run_phylobayes.sh`


```bash
#!/bin/bash
# ============================================================
# Script: run_phylobayes.sh (EXAMPLE TEMPLATE)
# Purpose: Bayesian phylogenetic inference using PhyloBayes MPI
#          under the CAT-GTR model as a sensitivity analysis.
#          This follows the procedure described in Methods:
#          "Topology testing and Bayesian inference".
# Stage: phylogenomics
# Author: Weili Lin
# Date: 2026-07-07
# Software: PhyloBayes MPI
# Note: This is an EXAMPLE template only.
# ============================================================

set -euo pipefail

# ------------------ Configuration (PLEASE MODIFY) ------------------
WORK_DIR="/path/to/your/project"
SORTWARE_DIR="/path/to/your/software"
ALIGNMENT="${WORK_DIR}/data/alignments/supermatrix.faa"
OUTPUT_DIR="${WORK_DIR}/results/phylobayes"
CHAIN_PREFIX="CAT_GTR"

mkdir -p "${OUTPUT_DIR}"

echo "=== [EXAMPLE] Starting PhyloBayes MPI analysis ==="
echo "Alignment file : ${ALIGNMENT}"
echo "Output directory: ${OUTPUT_DIR}"
echo "Model          : CAT-GTR"
echo "Number of chains: 10 (independent)"
date
echo ""

# ============================================================
# Step 1: Run 10 independent Markov chains
# ============================================================
echo ">>> [Step 1] Running 10 independent PhyloBayes chains under CAT-GTR model..."

# In the actual study, 10 independent chains were run in parallel.
# Each chain was run for a sufficient number of generations until
# topology frequencies stabilized (see Extended Data Table 5).

for i in $(seq 1 10); do
    echo "    Starting chain ${i}..."
    
    # Example command (commented out):
    mpirun -np 54 ${SORTWARE_DIR}/pb_mpi \
		-d "${ALIGNMENT}" \
		-cat -gtr \
        "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain${i}"
    
    echo "    (Chain ${i} - placeholder)"
done

echo ""

# ============================================================
# Step 2: Summarize chain behaviour and topology frequencies
# ============================================================
echo ">>> [Step 2] Summarizing topology frequencies across chains..."

# bpcomp is used to compare bipartition frequencies between chains
# and to assess convergence of the posterior distribution.

# Example command (commented out):
bpcomp -c 0.5 \
   -o "${OUTPUT_DIR}/${CHAIN_PREFIX}_bpcomp" \
   "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain1" \
   "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain2" \
   "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain3" \
   "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain4" \
   "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain5" \
   "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain6" \
   "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain7" \
   "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain8" \
   "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain9" \
   "${OUTPUT_DIR}/${CHAIN_PREFIX}_chain10"

echo "    (bpcomp summary - placeholder)"
echo ""

# ============================================================
# Completion
# ============================================================
echo "=== [EXAMPLE] PhyloBayes MPI analysis template completed ==="
echo ""
echo "See Methods section: 'Topology testing and Bayesian inference'"
echo "and Extended Data Table 5 for detailed chain-level results."
date
```

## B7 — Independent post-hoc audit

### Purpose
Second tool stack on **already clean** GS genomes; confirm residual contamination does not change the phylogenomic topology.

### Inputs
Clean FASTA for balanced collections in **Supplementary Tables 3–5**.

### Outputs

| Output | Where |
| --- | --- |
| Extra contigs removed by class | **Supplementary Table 7** (e.g. Asgard additional total ~0.69%) |
| Ultra-clean trees | `data/trees/maximum_likelihood/*ultra-clean*` → **Fig. 3a** |

### Script
`scripts/independent_audit/independent_audit.sh`


```bash
#!/bin/bash
# ============================================================
# Script: independent_audit.sh (EXAMPLE TEMPLATE)
# Purpose: Independent post-hoc contamination audit on the 
#          already decontaminated genome collection (GS5-clean).
#          This is a robustness verification step to confirm that
#          residual contamination does not affect the main conclusions.
#          This follows the procedure described in Methods:
#          "Independent post hoc contamination audit".
# Stage: robustness
# Author: Weili Lin
# Date: 2026-07-07
# Note: This is an EXAMPLE template only.
# ============================================================

set -euo pipefail

# ------------------ Configuration (PLEASE MODIFY) ------------------
WORK_DIR="/path/to/your/project"
SORTWARE_DIR="/path/to/your/software"
INPUT_GENOMES="${WORK_DIR}/data/genomes/GS5_clean"           # Already decontaminated genomes
OUTPUT_DIR="${WORK_DIR}/results/independent_audit"
LOG_DIR="${OUTPUT_DIR}/logs"

mkdir -p "${OUTPUT_DIR}/gunc" \
         "${OUTPUT_DIR}/virsorter2" \
         "${OUTPUT_DIR}/checkv" \
         "${OUTPUT_DIR}/whokaryote" \
         "${OUTPUT_DIR}/gc_anomaly" \
         "${LOG_DIR}"

echo "=== [EXAMPLE] Starting Independent Post-hoc Contamination Audit ==="
echo "Input genomes (GS5-clean) : ${INPUT_GENOMES}"
echo "Output directory          : ${OUTPUT_DIR}"
date
echo ""

# ============================================================
# GUNC - Detect prokaryotic chimerism and taxonomic inconsistency
# ============================================================
echo ">>> Running GUNC for chimerism and taxonomic inconsistency detection..."
# GUNC is used to detect potential bacterial contamination and chimerism.

gunc run \
	--file_suffix .fna \
	--threads 96 \
	--input_dir "${INPUT_GENOMES}" \
    -r ${SORTWARE_DIR}/gunc_db_gtdb214_2/gunc_db_gtdb214.dmnd \
	--out_dir --out_dir "${OUTPUT_DIR}/gunc" \
    --contig_taxonomy_output


echo "    (GUNC step - placeholder)"
echo ""

# ============================================================
# VirSorter2 + CheckV - Detect free viral sequences
# ============================================================
echo ">>> Running VirSorter2 + CheckV for free viral detection..."
# Only free virus (not provirus) is considered for removal.
# Proviral sequences are retained by default.

virsorter run \
	-w "${OUTPUT_DIR}/virsorter2" \
    -i "${INPUT_GENOMES}" \
	--include-groups dsDNAphage,NCLDV,RNA,ssDNA,lavidaviridae \
	-j 18 all \
	--db-dir ${SORTWARE_DIR}/VirSorter2-master/db

checkv end_to_end \
	"${INPUT_GENOMES}" \
	"${OUTPUT_DIR}/checkv" \
	-t 4

echo "    (VirSorter2 + CheckV step - placeholder)"
echo ""

# ============================================================
# Whokaryote - Detect eukaryotic contigs
# ============================================================
echo ">>> Running Whokaryote for eukaryotic sequence detection..."
# Used to identify potential eukaryotic contamination.

whokaryote.py \
	--contigs "${INPUT_GENOMES}" \
    --outdir "${OUTPUT_DIR}/whokaryote" \
	--model S

echo "    (Whokaryote step - placeholder)"
echo ""
```

---

# Part C — ESP inventories (functional complement to the trees)

**Question:** Does the same contamination bias that destabilizes trees also inflate eukaryotic signature protein (ESP) counts?

## C1 — ESP / iESP detection before vs after decontamination

### Inputs
Protein sets from the surveyed MAGs; ESP/iESP reference (e.g. Köstlbacher et al.).

### Outputs
`data/ESP/` → **Supplementary Tables 10–17**; **Fig. 4**  
(Asgard contributes the largest number of contamination-sensitive ESPs; most ESPs remain after cleaning.)

### Script
`scripts/ESP/run_ESP_pipeline.sh`


```bash
#!/usr/bin/env bash
# run_ESP_pipeline.sh
# ESP/iESP detection per genome:
#   DIAMOND screen → per-family hmmbuild → combine HMMs → hmmpress → hmmsearch
# Reference ESP/iESP families: curated list (e.g. Köstlbacher et al.)
# Contaminant-contig filtering (CAT/geNomad) and before/after counts are done downstream.

set -euo pipefail

########################
# Paths (edit these)
########################
WORKDIR="/path/to/your/project"
PROTEIN_DIR="${WORKDIR}/proteins"                 # one .faa per genome
ESP_REF_FAA="${WORKDIR}/reference/ESP_reference.faa"
ALN_DIR="${WORKDIR}/reference/cogs_v2"            # one *.aln per ESP family
ESP_HMM_DIR="${WORKDIR}/reference/ESP_iESP"       # output: one *.hmm per family
ESP_HMM_ALL="${WORKDIR}/reference/all_families.hmm"
THREADS="${THREADS:-16}"
THREADS_HMM="${THREADS_HMM:-32}"

mkdir -p "${WORKDIR}"/{diamond/per_genome,hmmsearch/per_genome}
mkdir -p "${ESP_HMM_DIR}"

########################
# 1. DIAMOND database
########################
echo "[1/5] Building DIAMOND database..."
diamond makedb \
  --in "${ESP_REF_FAA}" \
  -d "${WORKDIR}/ESP_reference" \
  --threads "${THREADS}"

########################
# 2. DIAMOND screen (per genome)
########################
echo "[2/5] DIAMOND screening (per genome)..."
for faa in "${PROTEIN_DIR}"/*.faa; do
  [[ -e "$faa" ]] || continue
  base=$(basename "$faa" .faa)
  echo "  → ${base}"

  diamond blastp \
    -q "$faa" \
    -d "${WORKDIR}/ESP_reference.dmnd" \
    -o "${WORKDIR}/diamond/per_genome/${base}_vs_ESP.tsv" \
    --ultra-sensitive \
    --evalue 1e-5 \
    --max-target-seqs 50 \
    --threads "${THREADS}" \
    --outfmt 6 qseqid sseqid pident length qlen slen qcovhsp scovhsp evalue bitscore

  awk 'BEGIN{FS=OFS="\t"} $9<=1e-5 && $7>=50 && $8>=50' \
    "${WORKDIR}/diamond/per_genome/${base}_vs_ESP.tsv" \
    > "${WORKDIR}/diamond/per_genome/${base}_vs_ESP.filtered.tsv"
done

# Optional merged table for downstream R summary
cat "${WORKDIR}/diamond/per_genome"/*_vs_ESP.filtered.tsv \
  > "${WORKDIR}/diamond/all_vs_ESP.filtered.tsv"

########################
# 3. Per-family HMMs from alignments
########################
echo "[3/5] hmmbuild (one HMM per family alignment)..."
for aln in "${ALN_DIR}"/*.aln; do
  [[ -e "$aln" ]] || continue
  base=$(basename "$aln" .aln)
  echo "  → ${base}"
  hmmbuild --cpu "${THREADS_HMM}" \
    "${ESP_HMM_DIR}/${base}.hmm" \
    "$aln"
done

########################
# 4. Combine HMMs + hmmpress
########################
echo "[4/5] Concatenating family HMMs and hmmpress..."
rm -f "${ESP_HMM_ALL}" "${ESP_HMM_ALL}".h3{m,i,f,p}

find "${ESP_HMM_DIR}" -name "*.hmm" -type f -print0 \
  | xargs -0 cat >> "${ESP_HMM_ALL}"

hmmpress "${ESP_HMM_ALL}"

########################
# 5. hmmsearch (per genome)
########################
echo "[5/5] hmmsearch (per genome)..."
for faa in "${PROTEIN_DIR}"/*.faa; do
  [[ -e "$faa" ]] || continue
  base=$(basename "$faa" .faa)
  echo "  → ${base}"

  hmmsearch --cpu "${THREADS}" \
    --tblout "${WORKDIR}/hmmsearch/per_genome/${base}_vs_ESP.tbl" \
    "${ESP_HMM_ALL}" \
    "$faa" \
    > "${WORKDIR}/hmmsearch/per_genome/${base}_vs_ESP.out"
done

echo "Done."
echo "  DIAMOND filtered hits: ${WORKDIR}/diamond/per_genome/ and all_vs_ESP.filtered.tsv"
echo "  HMMER tblout:          ${WORKDIR}/hmmsearch/per_genome/"
echo "  Next: map hits to contigs, exclude contaminant contigs (CAT/geNomad),"
echo "        then summarize Genome_Count_Before / After per ESP family."
```

## Figure–data map

Full panel notes: `scripts/figure_reproduction/figure_reproduction_README.md`


# Figure reproduction notes

This folder documents how main-figure panels were produced.

**Numerical results are defined by the deposited tables and tree files.**  
Display-only steps (iTOL, Adobe Illustrator, GraphPad Prism, BioRender) are described so that each panel can be matched to its source data. Pixel-identical layout is not required for verification.

Paths below are relative to the repository root unless noted.

---

## Scripted panels (Python)

| Panel | Script | Primary data source |
| --- | --- | --- |
| Fig. 1a | `fig1a_contamination_frequency.py` | `data/decontamination/assessment/` (lineage-level contamination frequencies) |
| Fig. 1c | `fig1c_contamination_derived_eukaryote_like_proteins.py` | `data/decontamination/assessment/contamination_derived_eukaryotic_like_proteins_by_Asgard_lineage.csv` (or equivalent `.xlsx`) |
| Fig. 3d | `fig3d_au_topology_test.py` | `data/trees/AU_tests/` (AU test output; *P*-values used in the panel) |

**Dependencies**

```bash
pip install matplotlib numpy
```

**Run** (from this directory, or adjust output paths):

```bash
python fig1a_contamination_frequency.py
python fig1c_contamination_derived_eukaryote_like_proteins.py
python fig3d_au_topology_test.py
```

Scripts may embed summary percentages for convenience; values should match the deposited assessment tables.

---

## Manually assembled panels

### Fig. 1b

- **Software:** GraphPad Prism  
- **Data:** `data/decontamination/assessment/` (Asgard lineage-level contamination frequencies, e.g. `contamination_by_Asgard_MAG_lineage` tables)  
- **Note:** Summary frequencies plotted in Prism. No custom analysis script.

### Fig. 2 (topology panels)

- **Topology source:** `data/trees/maximum_likelihood/*.contree`  
- **Display:** iTOL, then finalized in Adobe Illustrator (colours, labels, layout)  
- **Note:** Branching order and ultrafast bootstrap support are defined by the deposited `.contree` files. No topological editing beyond graphical presentation.

**Suggested panel–dataset correspondence** (aligned with the factorial design in the manuscript):

| Panels | Content |
| --- | --- |
| Fig. 2a–d | GS-Zhang2025-raw × each PMS (contamination present; sampling imbalanced) |
| Fig. 2e–h | GS-Zhang2025-B-raw × each PMS (contamination present; sampling balanced) |
| Fig. 2i–l | GS-Zhang2025-clean × each PMS (decontaminated; sampling imbalanced) |
| Fig. 2m–p | GS-Zhang2025-B-clean × each PMS (full control: decontaminated and sampling balanced) |

Exact file names follow the `GS-*-raw|clean_*_PMS-*.contree` convention under `data/trees/maximum_likelihood/`.

### Fig. 3a–c

#### Fig. 3a

- **Content:** Maximum-likelihood robustness using an independently audited ultra-clean genome collection.  
- **Topology source:** IQ-TREE consensus trees from the post-hoc audited (ultra-clean) dataset, e.g.  
  `data/trees/maximum_likelihood/` files matching `*ultra-clean*` and the GS × PMS combination shown in the panel.  
- **Supporting tables:** `data/trees/robustness_analyses.xlsx`; post-hoc audit summaries under `data/decontamination/phylogenomic_sets/` when deposited.  
- **Display:** iTOL → Adobe Illustrator.  
- **Note:** Branching order and support values are defined by the deposited `.contree` files.

#### Fig. 3b

- **Content:** Maximum-likelihood analyses under expanded archaeal taxon sampling.  
- **Topology source:** IQ-TREE consensus trees from expanded-sampling collections, e.g.  
  `data/alignments/GS-Zhang2025-B-clean_PMS-MediumMAG_expanded.faa` and the corresponding `.contree` under `data/trees/maximum_likelihood/`.  
- **Display:** iTOL → Adobe Illustrator.  
- **Note:** Topology is read from deposited tree files, not from the illustration.

#### Fig. 3c

- **Content:** Maximum-likelihood analysis under the alternative site-heterogeneous PMSF model.  
- **Topology source:** `data/trees/PMSF/` (files matching the GS × PMS combination shown in the panel).  
- **Display:** iTOL → Adobe Illustrator.

**Note (Fig. 3a–c):** Deposited `.contree` files are authoritative for branching order and nodal support. Illustrator was used only for schematic simplification, labelling and layout.

### Fig. 3e

- **Software:** GraphPad Prism  
- **Data:** Bayesian chain-level topology summaries (`data/trees/robustness_analyses.xlsx`; trees under `data/trees/CAT-GTR/`)  
- **Note:** Plotted in Prism from deposited summary counts (e.g. fraction of independent chains recovering a given placement). No custom script.

### Fig. 4a

- **Software:** GraphPad Prism  
- **Data:** `data/ESP/ESP_*_before_after_decontamination.xlsx`  
- **Note:** Counts of contamination-sensitive ESPs by archaeal group plotted in Prism.

### Fig. 4b–c

- **Software:** GraphPad Prism (quantitative elements); BioRender where schematic icons are used  
- **Data:** `data/ESP/` before/after tables  
- **Note:** Numerical values from deposited ESP tables; schematic elements from BioRender where applicable; panels assembled for publication layout.

---

## Extended Data and Supplementary figures (selected)

| Figure | Software | Data / notes |
| --- | --- | --- |
| Extended Data Fig. 6 (PMS construction workflow) | BioRender | Conceptual schematic; marker counts and overlap in `data/PMS/PMS_composition_and_overlap.xlsx` |
| Supplementary Fig. 1 (contamination vs assembly/quality metrics) | GraphPad Prism | Point-level and regression summaries intended for `data/decontamination/assessment/` (e.g. MAG contamination vs N50, CheckM metrics) |
| Extended Data figures on Asgard order/class contamination or contamination-derived eukaryotic-like proteins | GraphPad Prism | `data/decontamination/assessment/` (lineage-level tables, including `contamination_derived_eukaryotic_like_proteins_by_Asgard_lineage`) |
| Set overlaps for viral detection | R (`plot_extended_data_fig1_venn.R`) or Prism | `data/viral_detection_comparison/summary_venn_counts.xlsx` |

BioRender is used for schematic illustration only; it is not required to regenerate numerical panels. A journal-style attribution (e.g. “Created with BioRender.com”) may be placed in Acknowledgements or Methods if required by the publisher.

---

## General principles

1. **Trees:** Always prefer deposited `.contree` / PhyloBayes tree files over illustrated topologies.  
2. **Contamination and ESP counts:** Prefer tables under `data/decontamination/assessment/` and `data/ESP/`.  
3. **Scripts in this folder** reproduce selected bar/scatter-style panels; multi-panel phylogenies and most Prism figures are documented rather than fully scripted.  
4. File names in the table above may use `.xlsx` or `.csv` interchangeably if both formats are deposited; column meanings are described in `data/data_README.md`.
```

---

## End-to-end map (Part A vs Part B)

```text
Public MAGs (accessions in Supp. Tables 1–5)
        │
        ├──────────────────────────────────────────────┐
        │ Part A — contamination landscape             │
        │  CAT + geNomad on broad MAG sets             │
        │  → data/decontamination/assessment/          │
        │  → Fig. 1 (frequencies, Asgard lineages,     │
        │     contaminant eukaryote-like proteins)     │
        └──────────────────────────────────────────────┘
        │
        ▼
Part B — phylogenomics
  B1 Decontaminate each GS  → GS-*-clean;
       contigs removed % → Ext. Data Table 4
  B2 Balance sampling       → GS-*-B; Supp. Tables 3–5, 18–20
  B3 Four PMSs              → Ext. Data Table 1; data/PMS/
  B4 ML factorial trees     → Fig. 2; Supp. Table 6
  B5 AU tests               → Fig. 3d
  B6 CAT-GTR (10 chains)    → Fig. 3e; Ext. Data Table 5
  B7 Post-hoc audit         → Fig. 3a; Supp. Table 7
        │
        ▼
Part C — ESP before/after   → Fig. 4; Supp. Tables 10–17
```

### Main phylogenomic result (Part B, full control)
All **12** decontaminated × balanced genome-set–marker-set analyses place eukaryotes as sister to a monophyletic **TACK–Asgard** radiation, outside currently sampled Asgard subgroups.


In [ ]:
from pathlib import Path
checks = [
    "data/decontamination/assessment",
    "data/decontamination/phylogenomic_sets",
    "data/genome_sets",
    "data/PMS",
    "data/alignments",
    "data/trees",
    "data/ESP",
    "scripts/decontamination/run_decontamination.sh",
    "scripts/balancing/hierarchical_balancing_TACK_GS-ThisStudy-Balanced.R",
    "scripts/phylogenomics/run_iqtree_ml.sh",
    "scripts/independent_audit/independent_audit.sh",
    "scripts/ESP/run_ESP_pipeline.sh",
]
print("Presence check (repo root):")
for f in checks:
    print(f"  [{'OK' if Path(f).exists() else 'missing':7s}] {f}")
